## Практика программирования на GPU. Основы Triton

In [2]:
import torch
import triton
import triton.language as tl

# Определяем Triton кернел
@triton.jit
def add_kernel(
    x_ptr,          # Указатель на первый входной тензор
    y_ptr,          # Указатель на второй входной тензор  
    output_ptr,     # Указатель на выходной тензор
    n_elements,     # Количество элементов для обработки
    BLOCK_SIZE: tl.constexpr,  # Размер блока (значение известно при компиляции)
):
    # Каждая программа (блок) вычисляет свой диапазон индексов
    pid = tl.program_id(axis=0)  # Получаем ID программы в 1D сетке
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    
    # Создаём маску, чтобы не выйти за границы массива
    mask = offsets < n_elements
    
    # Загружаем данные из глобальной памяти
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    
    # Вычисляем сумму
    output = x + y
    
    # Сохраняем результат обратно в глобальную память
    tl.store(output_ptr + offsets, output, mask=mask)

# Функция-обёртка для запуска кернела
def add_vectors(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # Проверяем, что тензоры на GPU и имеют одинаковую форму
    assert x.is_cuda and y.is_cuda, "Тензоры должны быть на GPU"
    assert x.shape == y.shape, "Тензоры должны иметь одинаковую форму"
    
    # Создаём выходной тензор такой же формы
    output = torch.empty_like(x)
    n_elements = output.numel()
    
    # Определяем размер блока (желательно степень двойки)
    BLOCK_SIZE = 1024
    
    # Вычисляем количество блоков для покрытия всех элементов
    grid = (triton.cdiv(n_elements, BLOCK_SIZE),)
    
    # Запускаем кернел
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE)
    
    return output

# Тестирование функции
if __name__ == "__main__":
    # Инициализируем конкретные векторы для примера
    size = 10000
    print(f"Создаём векторы размером {size} элементов...")
    
    # Создаём тестовые данные на CPU
    x_cpu = torch.arange(1, size + 1, dtype=torch.float32)  # [1, 2, 3, ..., 10000]
    y_cpu = torch.ones(size, dtype=torch.float32) * 5       # [5, 5, 5, ..., 5]
    
    # Перемещаем данные на GPU
    x_gpu = x_cpu.cuda()
    y_gpu = y_cpu.cuda()
    
    print("Вектор x (первые 10 элементов):", x_cpu[:10].tolist())
    print("Вектор y (первые 10 элементов):", y_cpu[:10].tolist())
    
    # Вычисляем сумму с помощью Triton
    print("\nВычисляем сумму с помощью Triton...")
    result_triton = add_vectors(x_gpu, y_gpu)
    
    # Вычисляем сумму с помощью PyTorch для проверки
    print("Вычисляем сумму с помощью PyTorch для проверки...")
    result_pytorch = x_gpu + y_gpu
    
    # Проверяем корректность
    print("\nПроверяем корректность...")
    is_correct = torch.allclose(result_triton, result_pytorch, atol=1e-6)
    print(f"Результаты совпадают: {is_correct}")
    
    # Показываем результаты
    print(f"\nРезультат (первые 10 элементов): {result_triton.cpu()[:10].tolist()}")
    print(f"Ожидаемый результат: {[6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]}")
    
    # Бенчмарк производительности
    print("\nЗапускаем бенчмарк...")
    import time
    
    # Тёплый запуск
    for _ in range(10):
        _ = add_vectors(x_gpu, y_gpu)
        _ = x_gpu + y_gpu
    
    # Измеряем время Triton
    start_time = time.time()
    for _ in range(100):
        result_triton = add_vectors(x_gpu, y_gpu)
    torch.cuda.synchronize()
    triton_time = time.time() - start_time
    
    # Измеряем время PyTorch
    start_time = time.time()
    for _ in range(100):
        result_pytorch = x_gpu + y_gpu
    torch.cuda.synchronize()
    pytorch_time = time.time() - start_time
    
    print(f"Triton время: {triton_time:.4f} сек")
    print(f"PyTorch время: {pytorch_time:.4f} сек")
    print(f"Отношение скоростей: {pytorch_time/triton_time:.2f}x")

Создаём векторы размером 10000 элементов...
Вектор x (первые 10 элементов): [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
Вектор y (первые 10 элементов): [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]

Вычисляем сумму с помощью Triton...
Вычисляем сумму с помощью PyTorch для проверки...

Проверяем корректность...
Результаты совпадают: True

Результат (первые 10 элементов): [6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]
Ожидаемый результат: [6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]

Запускаем бенчмарк...
Triton время: 0.0027 сек
PyTorch время: 0.0008 сек
Отношение скоростей: 0.31x


### Задание 1
По аналогии с примером выше реализуйте кернел для скалярного произведения двух векторов. 

Используйте метод `tl.atomic_add`, который суммирует результаты с разных программ перед возвращением финального результата. Подробнее о методе — в [официальной документации Triton](https://triton-lang.org/main/python-api/triton.language.html).

In [7]:
import torch
import triton
import triton.language as tl

@triton.jit
def dot_product_simple_kernel(
    x_ptr,          # Указатель на первый вектор
    y_ptr,          # Указатель на второй вектор
    output_ptr,     # Указатель на выход (один элемент)
    n_elements,     # Количество элементов
    BLOCK_SIZE: tl.constexpr,  # Размер блока
):
    # Получаем номер программы
    pid = tl.program_id(axis=0)

    # Вычисляем старт блока
    block_start = pid * BLOCK_SIZE
    # Вычисляем смещения для данного блока
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # Вычисляем маску для данного блока
    mask = offsets < n_elements
    
     # Загружаем данные
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    
    # вычисляем скалярное произведение в блоке
    # Самый простой способ - использовать tl.sum
    block_sum = tl.sum(x * y, axis=0)
    
    # Атомарная редукция между блоками
    tl.atomic_add(output_ptr, block_sum)

# Функция-обёртка с полной GPU редукцией
def dot_product_full_gpu(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda, "Тензоры должны быть на GPU"
    assert x.shape == y.shape, "Тензоры должны иметь одинаковую форму"
    assert x.dim() == 1, "Тензоры должны быть векторами"
    
    n_elements = x.numel()
    BLOCK_SIZE = 1024
    num_blocks = triton.cdiv(n_elements, BLOCK_SIZE)
    
    # Создаём выходной тензор из одного элемента
    result = torch.zeros(1, dtype=torch.float32, device='cuda')
    
    # Запускаем кернел
    dot_product_simple_kernel[(num_blocks,)](x, y, result, n_elements, BLOCK_SIZE)
    
    return result.cpu()



In [8]:
size = 10000
x = torch.arange(1, size + 1, dtype=torch.float32, device='cuda')
y = torch.ones(size, dtype=torch.float32, device='cuda') * 2

print("Тестирование кернела скалярного произведения...")

# Тестируем разные версии
result_simple = dot_product_full_gpu(x, y)
result_pytorch = torch.dot(x, y).cpu()

print(f"Triton результат: {result_simple.item():.1f}")
print(f"PyTorch результат: {result_pytorch.item():.1f}")
print(f"Совпадение: {torch.allclose(result_simple, result_pytorch, atol=1e-4)}")

# Бенчмарк
import time

print("\nБенчмарк:")
start = time.time()
for _ in range(100):
    result = dot_product_full_gpu(x, y)
torch.cuda.synchronize()
triton_time = time.time() - start

start = time.time()
for _ in range(100):
    result = torch.dot(x, y)
torch.cuda.synchronize()
pytorch_time = time.time() - start

print(f"Triton: {triton_time:.4f}с")
print(f"PyTorch: {pytorch_time:.4f}с")
print(f"Отношение: {pytorch_time/triton_time:.2f}x")


Тестирование кернела скалярного произведения...
Triton результат: 100010000.0
PyTorch результат: 100010000.0
Совпадение: True

Бенчмарк:
Triton: 0.0067с
PyTorch: 0.0015с
Отношение: 0.22x
